In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*Szacowane wykorzystanie zasobów: jedna minuta na procesorze Eagle (UWAGA: to tylko szacunek. Twój czas wykonania może się różnić.)*

## Tło

Circuit-knitting to termin zbiorczy, który obejmuje różne metody podziału obwodu na wiele mniejszych podobwodów zawierających mniej bramek i/lub qubitów. Każdy z podobwodów może być wykonywany niezależnie, a końcowy wynik uzyskuje się przez klasyczne post-przetwarzanie wyników każdego podobwodu. Technika ta jest dostępna w [dodatku Qiskit do cięcia obwodów](https://qiskit.github.io/qiskit-addon-cutting/index.html); szczegółowe wyjaśnienie techniki znajduje się w [dokumentacji](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) wraz z innymi [materiałami wprowadzającymi](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html).

Ten notebook dotyczy metody zwanej <b>cięciem przewodów</b>, w której obwód jest dzielony wzdłuż przewodu [\[1\], \[2\]](#references). Warto zauważyć, że podział jest prosty w przypadku obwodów klasycznych, ponieważ wynik w miejscu podziału można wyznaczyć deterministycznie — jest to 0 lub 1. Jednak stan qubitu w miejscu cięcia jest, ogólnie rzecz biorąc, stanem mieszanym. Dlatego każdy podobwód musi być mierzony wielokrotnie w różnych bazach (zazwyczaj tomograficznie pełnych zbiorach baz, takich jak baza Pauliego [\[3\], \[4\]](#references)) i odpowiednio przygotowany w swoim stanie własnym. Poniższy rysunek (<i>źródło: praca doktorska, Ritajit Majumdar</i>) pokazuje przykład cięcia przewodów dla 4-qubitowego stanu GHZ na trzy podobwody. Tutaj $M_j$ oznacza zestaw baz (zazwyczaj Pauli X, Y i Z), a $P_i$ oznacza zestaw stanów własnych (zazwyczaj $|0\rangle$, $|1\rangle$, $|+\rangle$ i $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Ponieważ każdy podobwód ma mniej qubitów i/lub bramek, oczekuje się, że będzie mniej podatny na szumy. Ten notebook pokazuje przykład, gdzie ta metoda może być użyta do efektywnego tłumienia szumów w systemie.

## Wymagania

Przed rozpoczęciem tego samouczka upewnij się, że masz zainstalowane następujące elementy:

- Qiskit SDK v2.0 lub nowszy, z obsługą [wizualizacji](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 lub nowszy ( `pip install qiskit-ibm-runtime` )
- Dodatek Qiskit do cięcia obwodów v0.9.0 lub nowszy (`pip install qiskit-addon-cutting`)

W tym notebooku rozważymy obwód Many Body Localization (MBL). Obwód MBL jest obwodem efektywnym sprzętowo i jest sparametryzowany dwoma parametrami $\theta$ i $\vec{\phi}$. Gdy $\theta$ jest ustawione na $0$ i stan początkowy jest przygotowany w $|0\rangle$ dla wszystkich qubitów, idealna wartość oczekiwana $\langle Z_i \rangle$ wynosi $+1$ dla każdego miejsca qubitu $i$ niezależnie od wartości $\vec{\phi}$. Więcej szczegółów na temat obwodów MBL znajdziesz w <a href="https://arxiv.org/abs/2307.07552">tym artykule</a>.

## Konfiguracja

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Część I. Przykład w małej skali

### Krok 1: Odwzorowanie klasycznych danych wejściowych na problem kwantowy

Najpierw budujemy szablon obwodu bez żadnych konkretnych wartości parametrów. Dodajemy również miejsca zastępcze, zwane `CutWire`, aby oznaczyć pozycje cięć. W przykładzie małej skali rozważamy 10-qubitowy obwód MBL.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Teraz oznaczamy obwód do cięcia, wstawiając odpowiednie **CutWire**, aby utworzyć dwa w przybliżeniu równe cięcia. Ustawiamy `use_cut=True` w funkcji i pozwalamy jej na oznaczenie po $\frac{n}{2}$ qubitach, gdzie $n$ to liczba qubitów w oryginalnym obwodzie.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### Krok 2: Optymalizacja problemu pod kątem wykonania na sprzęcie kwantowym
Następnie dzielimy obwód na dwa mniejsze podobwody. W tym przykładzie ograniczamy się do tylko 2 podobwodów. W tym celu używamy <a href="https://qiskit.github.io/qiskit-addon-cutting/">Dodatku Qiskit: Cięcie Obwodów</a>.
#### Podziel obwód na mniejsze podobwody
Przecięcie przewodu w danym miejscu zwiększa liczbę qubitów o jeden. Oprócz oryginalnego qubitu pojawia się teraz dodatkowy qubit jako miejsce zastępcze w obwodzie po cięciu. Poniższy obraz pokazuje reprezentację:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

Ten Dodatek używa funkcji `cut_wires`, aby uwzględnić dodatkowe qubity powstałe w wyniku cięcia.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### Utwórz i rozszerz obserwable
Teraz konstruujemy obserwablę $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$. Ponieważ idealna wartość $\langle Z_i \rangle$ dla każdego $i$ wynosi $+1$, idealna wartość $M_z$ również wynosi $+1$.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Jednak zauważ, że liczba qubitów w obwodzie wzrosła po wstawieniu wirtualnych 2-qubitowych operacji `Move` po cięciu. Dlatego musimy również rozszerzyć obserwable, wstawiając tożsamości, aby dostosować je do bieżącego obwodu.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Zwizualizujmy podobwody

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

Obserwable zostały również podzielone, aby pasowały do podobwodów

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

Zauważ, że każdy podobwód prowadzi do pewnej liczby próbek. Rekonstrukcja uwzględnia wynik każdej z tych próbek. Każda z tych próbek jest określana jako `subexperiment`.
Rozszerzenie obserwabli za pomocą operacji `Move` wymaga struktury danych `PauliList`. Możemy też utworzyć obserwablę $M_z$ w bardziej ogólnej strukturze danych `SparsePauliOp`, która będzie przydatna później podczas rekonstrukcji podeksperymentów.

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Zobaczmy dwa przykłady, w których qubity cięcia są mierzone w dwóch różnych bazach. Najpierw jest mierzony w normalnej bazie Z, a następnie w bazie X.

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### Transpiluj każdy podeksperyment

Obecnie musimy transpilować nasze obwody przed ich przesłaniem do wykonania. Dlatego najpierw transpilujemy każdy obwód w podeksperymentach.